# ForgeryGuard AI — M1 | Data Preparation & Validation


**Kaggle-native notebook** · Dataset: DEFACTO Copy-Move + MSCOCO originals · 18,194 original/forged pairs

هذا الملف يستعيد ويتحقق من مخرجات M1 السابقة الموجودة في Kaggle، دون إعادة تقسيم البيانات أو تعديل مصادرها. يستخدم `/kaggle/input` للقراءة و`/kaggle/working` للتقارير الجديدة.

This notebook restores and validates the existing M1 deliverables in Kaggle. It does not regenerate the saved split or overwrite the source datasets. Read-only inputs are under `/kaggle/input`; new reports go to `/kaggle/working`.

**Before running:** Add Input → `mkajrfi/forgeryguard-ai` and `defactodataset/defactocopymove`. No Drive, API tokens, or large downloads are required when inputs are attached.

**Scope:** Data validation and reproducibility; license/privacy checks remain manual and pending until verified.

## M1 — Step 1: Environment & inputs

التحقق من بيئة Kaggle والمجموعات المرتبطة، دون تنزيل ملفات.

Verify Kaggle input datasets without downloading files.

In [ ]:
from pathlib import Path
import os, sys, json, csv, random, hashlib
import pandas as pd
import numpy as np
from PIL import Image

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/ForgeryGuard_AI_M1')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
if not INPUT_ROOT.is_dir():
    raise RuntimeError('Run this notebook in Kaggle with both datasets attached.')
print('Python:', sys.version.split()[0])
print('Input datasets:', [p.name for p in INPUT_ROOT.iterdir() if p.is_dir()])
print('Working directory:', WORK_ROOT)

## M1 — Step 2: Locate attached datasets

نحدد مجلدَي المشروع وDEFACTO من المسارات الفعلية.

Resolve the actual attached dataset paths.

In [ ]:

from pathlib import Path

# تحديد مجلد البيانات في كاجل
INPUT_ROOT = Path("/kaggle/input")

if not INPUT_ROOT.is_dir():
    raise FileNotFoundError("Kaggle input directory not found")

# عرض المجلدات المتاحة
print("Connected datasets:")
for folder in sorted(INPUT_ROOT.iterdir()):
    if folder.is_dir():
        print("-", folder.name)

# البحث عن مجموعة المشروع عبر ملف التقسيم
project_matches = list(
    INPUT_ROOT.rglob("dataset_split_portable.csv")
)

# البحث عن مجموعة الصور المزوّرة عبر مجلدها
defacto_matches = [
    p for p in INPUT_ROOT.rglob("copymove_img")
    if p.is_dir() and (p / "img").is_dir()
]

# التحقق من النتائج
if not project_matches:
    raise FileNotFoundError(
        "ForgeryGuard dataset is not attached or manifest is missing"
    )

if not defacto_matches:
    raise FileNotFoundError(
        "DEFACTO dataset is not attached or image folder is missing"
    )

# تحديد المسارات الفعلية
PROJECT_INPUT = project_matches[0].parent
DEFACTO_INPUT = defacto_matches[0].parent

print("\nProject input:", PROJECT_INPUT)
print("DEFACTO input:", DEFACTO_INPUT)
print("STATUS: DATASETS FOUND")


## M1 — Step 3: Locate M1 artifacts

نبحث عن ملفات M1 المحفوظة، دون افتراض مواضعها الدقيقة.

Locate the saved M1 artifacts.

In [ ]:
# البحث عن الملفات المرجعية في مجموعة المشروع
artifact_names = ['dataset_split_portable.csv', 'dataset_split.csv', 'training_manifest.csv',
                  'validated_image_pairs.csv', 'tampering_bounding_boxes.csv', 'dataset_card.json']
ARTIFACTS = {}
for name in artifact_names:
    matches = list(PROJECT_INPUT.rglob(name))
    ARTIFACTS[name] = matches[0] if matches else None
    print(name, '=>', ARTIFACTS[name] or 'NOT FOUND')
if ARTIFACTS['dataset_split_portable.csv'] is None:
    raise FileNotFoundError('dataset_split_portable.csv is required.')

## M1 — Step 4: Load the saved split

نقرأ التقسيم الموجود ونحافظ عليه كما هو.

Read and preserve the existing train/val/test split.

In [ ]:
MANIFEST = pd.read_csv(ARTIFACTS['dataset_split_portable.csv'], dtype=str)
required = {'source_group','split','original_filename','forged_filename'}
if not required.issubset(MANIFEST.columns):
    raise ValueError(f'Missing columns: {sorted(required-set(MANIFEST.columns))}')
print('Pairs:', len(MANIFEST))
print(MANIFEST['split'].value_counts().to_string())
print(MANIFEST.head(3).to_string(index=False))

## M1 — Step 5: Validate labels and counts

نتحقق من صحة أسماء التقسيم وعدد الأزواج المتوقع دون تعديله.

Validate split labels and expected counts without modifying the manifest.

In [ ]:
expected = {'train':12735,'val':2729,'test':2730}
counts = MANIFEST['split'].value_counts().to_dict()
assert counts == expected, f'Unexpected split counts: {counts}'
assert len(MANIFEST) == 18194, 'Unexpected number of image pairs'
assert MANIFEST[list(required)].notna().all().all(), 'Missing required values'
print('Split counts: PASS')

## M1 — Step 6: Check leakage by source group

نتحقق من عدم ظهور مجموعة المصدر نفسها في أكثر من تقسيم.

Ensure each source group belongs to only one split.

In [ ]:
group_splits = MANIFEST.groupby('source_group')['split'].nunique()
leaks = group_splits[group_splits > 1]
print('Source groups:', len(group_splits))
print('Leaking source groups:', len(leaks))
assert leaks.empty, 'Source-group leakage detected'
print('Group leakage check: PASS')

## M1 — Step 7: Check filenames and duplicates

نفحص تكرار أسماء الصور وارتباط كل زوج بمجموعة المصدر.

Check filename uniqueness and pair-to-source consistency.

In [ ]:
for col in ['original_filename','forged_filename']:
    print(col, 'unique:', MANIFEST[col].nunique(), 'duplicates:', MANIFEST[col].duplicated().sum())
assert MANIFEST['original_filename'].is_unique, 'Repeated original filenames'
assert MANIFEST['forged_filename'].is_unique, 'Repeated forged filenames'
assert (MANIFEST['original_filename'].str.replace('.jpg','',regex=False) == MANIFEST['source_group']).all(), 'Original filename mismatch'
print('Filename checks: PASS')

## M1 — Step 8: Find original image directory

نحدد الصور الأصلية المرفقة من Kaggle، بدون استخدام واجهة التنزيل.

Locate attached original images without Kaggle API calls.

In [ ]:
# التحقق من المسار المعروف أولًا ثم البحث في المجموعة
original_candidates = [PROJECT_INPUT/'forgeryguard_originals'/'Original', PROJECT_INPUT/'Original']
original_candidates += [p for p in PROJECT_INPUT.rglob('Original') if p.is_dir()]
ORIGINAL_DIR = next((p for p in original_candidates if p.is_dir() and any(p.glob('*.jpg'))), None)
if ORIGINAL_DIR is None:
    raise FileNotFoundError('Original images directory not found. Check the attached project dataset.')
print('Original images directory:', ORIGINAL_DIR)
print('JPG files:', sum(1 for _ in ORIGINAL_DIR.glob('*.jpg')))

## M1 — Step 9: Find DEFACTO images and masks

نحدد صور DEFACTO وأقنعة التلاعب في مجموعة البيانات المرفقة.

Locate DEFACTO forged images and probe masks.

In [ ]:
# البحث عن البنية المعروفة لمجموعة DEFACTO
forged_candidates = [p for p in DEFACTO_INPUT.rglob('img') if p.is_dir() and p.parent.name == 'copymove_img']
mask_candidates = [p for p in DEFACTO_INPUT.rglob('probe_mask') if p.is_dir()]
if not forged_candidates or not mask_candidates:
    raise FileNotFoundError('DEFACTO forged images or probe masks not found')
FORGED_DIR = forged_candidates[0]
MASK_DIR = mask_candidates[0]
print('Forged images:', FORGED_DIR)
print('Probe masks:', MASK_DIR)

## M1 — Step 10: Validate image availability

نطابق أسماء الملفات في تقسيم M1 مع الصور الموجودة فعليًا.

Match all saved split filenames to the attached images.

In [ ]:
missing_original = [x for x in MANIFEST['original_filename'] if not (ORIGINAL_DIR/x).is_file()]
missing_forged = [x for x in MANIFEST['forged_filename'] if not (FORGED_DIR/x).is_file()]
print('Missing originals:', len(missing_original))
print('Missing forged:', len(missing_forged))
if missing_original: print('Sample missing originals:', missing_original[:5])
if missing_forged: print('Sample missing forged:', missing_forged[:5])
assert not missing_original and not missing_forged, 'Missing image files'

## M1 — Step 11: Validate probe mask availability

نطابق قناع التلاعب مع اسم الصورة المزوّرة بعد تغيير الامتداد إلى JPG.

Verify that each forged image has a corresponding JPEG probe mask.

In [ ]:
# أقنعة DEFACTO من نوع JPEG وبنفس اسم الصورة دون الامتداد
MASK_PATHS = [MASK_DIR/(Path(x).stem+'.jpg') for x in MANIFEST['forged_filename']]
missing_masks = [str(p.name) for p in MASK_PATHS if not p.is_file()]
print('Missing probe masks:', len(missing_masks))
assert not missing_masks, f'Missing masks: {missing_masks[:5]}'

## M1 — Step 12: Inspect image and mask samples

نفتح عينات قليلة للتحقق من إمكانية القراءة وتوافق الأبعاد.

Inspect a small deterministic sample for readability and aligned dimensions.

In [ ]:
sample = MANIFEST.sample(n=min(12,len(MANIFEST)), random_state=251)
issues = []
for _, row in sample.iterrows():
    op = ORIGINAL_DIR/row['original_filename']
    fp = FORGED_DIR/row['forged_filename']
    mp = MASK_DIR/(Path(row['forged_filename']).stem+'.jpg')
    try:
        with Image.open(op) as im: original_size = im.size
        with Image.open(fp) as im: forged_size = im.size
        with Image.open(mp) as im: mask_size = im.size
        if forged_size != mask_size or original_size != forged_size:
            issues.append((row['source_group'], original_size, forged_size, mask_size))
    except Exception as e:
        issues.append((row['source_group'], type(e).__name__))
print('Sample checked:', len(sample))
print('Issues:', issues[:5])
assert not issues, 'Sample image/mask validation failed'

## M1 — Step 13: Inspect mask binary thresholds

نفحص عينة من الأقنعة ونستخدم حد 127 لأن الأقنعة بصيغة JPEG مضغوطة.

Inspect mask values using threshold 127 for JPEG-compressed masks.

In [ ]:
mask_stats = []
for _, row in sample.iterrows():
    mp = MASK_DIR/(Path(row['forged_filename']).stem+'.jpg')
    with Image.open(mp) as im:
        mask = np.asarray(im.convert('L')) > 127
    mask_stats.append({'source_group':row['source_group'], 'tampered_pixels':int(mask.sum()), 'total_pixels':int(mask.size)})
MASK_SAMPLE = pd.DataFrame(mask_stats)
print(MASK_SAMPLE.to_string(index=False))
assert (MASK_SAMPLE['tampered_pixels'] > 0).all(), 'Empty sample probe mask'

## M1 — Step 14: Inspect saved bounding boxes

نقرأ مربعات التلاعب المحفوظة من M1 ونفحص بنيتها دون إعادة استخراجها.

Inspect the saved bounding-box artifact without regenerating annotations.

In [ ]:
BBOX_PATH = ARTIFACTS['tampering_bounding_boxes.csv']
if BBOX_PATH is None:
    print('Saved bounding-box file is missing; M1 annotation artifact is incomplete.')
    BBOX = None
else:
    BBOX = pd.read_csv(BBOX_PATH)
    print('Bounding-box rows:', len(BBOX))
    print('Columns:', list(BBOX.columns))
    print(BBOX.head(3).to_string(index=False))

## M1 — Step 14 B: Visual Audit of Image Pairs and Annotations

Inspect 12 fixed training pairs: original image, forged image, binary manipulation mask, and saved bounding box. Check scene correspondence, mask alignment, annotation quality, and visible personal information. Record findings after reviewing the displayed images.

نراجع بصريًا 12 زوجًا ثابتًا من التدريب: الأصلية، المعدلة، قناع التلاعب، والمربع المحفوظ. نتحقق من تطابق المشهد، وصحة تحديد التلاعب، وأي معلومات شخصية ظاهرة. عرض الصور لا يعني اعتماد المراجعة تلقائيًا، ولا تعمم نتيجة العينة على كامل البيانات.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# التحقق من تشغيل خلايا M1 السابقة
needed = ["MANIFEST", "ORIGINAL_DIR", "FORGED_DIR", "MASK_DIR", "BBOX"]
missing = [name for name in needed if name not in globals()]
if missing:
    raise RuntimeError(f"Run previous M1 cells first. Missing: {missing}")

if BBOX is None:
    raise RuntimeError("The saved bounding-box table is missing.")

# عينة ثابتة من التدريب، دون استخدام صور الاختبار
visual_sample = (
    MANIFEST.loc[MANIFEST["split"] == "train"]
    .sample(n=12, random_state=251)
    .reset_index(drop=True)
)

# البحث باسم الملف لتجنب اختلاف تنسيق source_group
box_lookup = BBOX.groupby("forged_filename", sort=False)

# عرض 3 أزواج في كل لوحة
for start in range(0, len(visual_sample), 3):
    batch = visual_sample.iloc[start:start + 3]
    fig, axes = plt.subplots(
        len(batch), 4, figsize=(18, 4 * len(batch)), squeeze=False
    )

    for row_idx, (_, row) in enumerate(batch.iterrows()):
        with Image.open(ORIGINAL_DIR / row["original_filename"]) as im:
            original = np.asarray(im.convert("RGB"))

        with Image.open(FORGED_DIR / row["forged_filename"]) as im:
            forged = np.asarray(im.convert("RGB"))

        mask_name = Path(row["forged_filename"]).stem + ".jpg"
        with Image.open(MASK_DIR / mask_name) as im:
            mask = np.asarray(im.convert("L")) > 127

        group = row["source_group"]

        axes[row_idx, 0].imshow(original)
        axes[row_idx, 0].set_title(f"{group} | Original")

        axes[row_idx, 1].imshow(forged)
        axes[row_idx, 1].set_title(f"{group} | Forged")

        axes[row_idx, 2].imshow(mask, cmap="gray", vmin=0, vmax=1)
        axes[row_idx, 2].set_title("Binary mask | threshold > 127")

        # عرض القناع فوق الصورة مع المربعات المحفوظة
        ax = axes[row_idx, 3]
        ax.imshow(forged)

        overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
        overlay[mask] = [1.0, 0.0, 0.0, 0.4]
        ax.imshow(overlay)

        filename = row["forged_filename"]
        if filename in box_lookup.groups:
            for _, box in box_lookup.get_group(filename).iterrows():
                xmin, ymin, xmax, ymax = [
                    float(box[key])
                    for key in ["xmin", "ymin", "xmax", "ymax"]
                ]
                ax.add_patch(Rectangle(
                    (xmin, ymin),
                    xmax - xmin,
                    ymax - ymin,
                    fill=False,
                    edgecolor="lime",
                    linewidth=2
                ))
            ax.set_title("Mask overlay + saved box")
        else:
            ax.set_title("WARNING: saved box missing")

        for ax in axes[row_idx]:
            ax.axis("off")

    plt.tight_layout()
    plt.show()
    plt.close(fig)

print("Displayed source groups:")
print(visual_sample["source_group"].tolist())
print("\nVisual review pending. No files or splits changed.")

## M1 — Step 15: Inspect other M1 manifests

نفحص مخرجات M1 الإضافية ونحتفظ بها كمرجع.

Inspect additional saved M1 manifests for reproducibility.

In [ ]:
for name in ['training_manifest.csv','validated_image_pairs.csv','dataset_split.csv']:
    p = ARTIFACTS[name]
    if p is None:
        print(name, ': NOT FOUND')
    else:
        df = pd.read_csv(p, nrows=3)
        print(name, ':', list(df.columns))

## M1 — Step 16: Review dataset card

نقرأ بطاقة البيانات السابقة دون تغيير حالة التراخيص والخصوصية.

Read the saved dataset card without changing license or privacy review status.

In [ ]:
CARD_PATH = ARTIFACTS['dataset_card.json']
if CARD_PATH is None:
    print('Dataset card missing; manual review remains required.')
    DATASET_CARD = {}
else:
    with CARD_PATH.open('r', encoding='utf-8') as f: DATASET_CARD = json.load(f)
    print(json.dumps(DATASET_CARD, ensure_ascii=False, indent=2)[:3500])
print('Reminder: license and privacy review must be confirmed manually.')

## M1 — Step 17: Build classification paths

نضيف مسارات الصور إلى نسخة عمل محلية، مع الحفاظ على تقسيم M1 الأصلي.

Build a working classification manifest without altering the original split.

In [ ]:
# إنشاء نسختين لكل زوج: أصلية ومزوّرة
original_rows = pd.DataFrame({'source_group':MANIFEST['source_group'], 'split':MANIFEST['split'],
    'label':0, 'class_name':'original', 'image_path':MANIFEST['original_filename'].map(lambda x:str(ORIGINAL_DIR/x))})
forged_rows = pd.DataFrame({'source_group':MANIFEST['source_group'], 'split':MANIFEST['split'],
    'label':1, 'class_name':'forged', 'image_path':MANIFEST['forged_filename'].map(lambda x:str(FORGED_DIR/x))})
CLASSIFICATION = pd.concat([original_rows,forged_rows],ignore_index=True)
print('Classification images:',len(CLASSIFICATION))
print(pd.crosstab(CLASSIFICATION['split'], CLASSIFICATION['class_name']))

## M1 — Step 18: Verify training readiness

نتحقق من توازن الفئات وسلامة المسارات قبل اعتماد البيانات للتدريب.

Validate class balance and file paths before training.

In [ ]:
assert len(CLASSIFICATION) == 36388
assert CLASSIFICATION['image_path'].map(lambda p:Path(p).is_file()).all()
balance = pd.crosstab(CLASSIFICATION['split'], CLASSIFICATION['label'])
assert (balance[0] == balance[1]).all(), 'Class imbalance in saved split'
assert CLASSIFICATION.groupby('source_group')['split'].nunique().max() == 1
print('M2/M3 classification readiness: PASS')

## M1 — Step 19: Export validated working manifests

نحفظ قوائم المسارات وتقارير التحقق في مساحة العمل فقط، دون تعديل ملفات الإدخال.

Save validated working manifests to the writable output directory only.

In [ ]:
CLASSIFICATION_PATH = WORK_ROOT/'classification_manifest.csv'
CLASSIFICATION.to_csv(CLASSIFICATION_PATH,index=False)
report = {'project':'ForgeryGuard AI','phase':'M1','pairs':len(MANIFEST),
    'classification_images':len(CLASSIFICATION), 'split_pairs':{k:int(v) for k,v in MANIFEST['split'].value_counts().items()},
    'source_groups':int(MANIFEST['source_group'].nunique()),
    'missing_originals':len(missing_original),'missing_forged':len(missing_forged),'missing_masks':len(missing_masks),
    'privacy_review':'PENDING — manual verification', 'license_review':'PENDING — manual verification'}
REPORT_PATH = WORK_ROOT/'m1_validation_report.json'
REPORT_PATH.write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
print('Classification manifest:', CLASSIFICATION_PATH)
print('Validation report:', REPORT_PATH)

## M1 — Step 20: Final M1 checkpoint

نعرض ملخص الإنجاز ونوضح ما يجب التحقق منه يدويًا قبل استخدام البيانات أو نشرها.

Summarize M1 validation and outstanding manual compliance checks.

In [ ]:
print('========== M1 FINAL CHECKPOINT ==========')
print('Pairs:', len(MANIFEST))
print('Classification images:', len(CLASSIFICATION))
print('Source-group leakage:', len(leaks))
print('Missing originals:', len(missing_original))
print('Missing forged:', len(missing_forged))
print('Missing masks:', len(missing_masks))
print('Classification readiness: READY')
print('Privacy review: PENDING')
print('License review: PENDING')
print('Output folder:', WORK_ROOT)
print('=========================================')

### Next: M2 and M3

استخدم ملف `classification_manifest.csv` الناتج من هذه الجلسة، أو شغّل خلية بناء المسارات نفسها في دفتر M2/M3. مسارات `/kaggle/input` تعمل عند ربط مجموعتي البيانات بالدفتر الجديد.

Use the generated classification manifest in this session, or recreate its path mapping in M2/M3 after attaching the same inputs. `/kaggle/input` paths are read-only; `/kaggle/working` outputs can be saved as Kaggle Notebook outputs.

**Important:** The notebook validates previously generated M1 deliverables; it does not claim to reproduce the original data acquisition, privacy screening, or annotation-generation process from scratch.